# Sigap.ai - Linear SVM Baseline for Sentiment Analysis

Notebook ini disusun untuk proyek capstone IBM SkillsBuild Sigap.ai sebagai baseline yang dapat langsung dijalankan di Google Colab.

Scope notebook ini dibatasi secara tegas pada data yang sudah final:
- proses data collection sudah selesai
- proses cleaning sudah selesai
- proses labeling sudah selesai
- proses balancing sudah selesai
- proses splitting sudah selesai
- proses preprocessing sudah selesai

Eksperimen ini hanya menggunakan:
- feature: `Review_Text_Processed`
- target: `Sentiment_Label`

Fokus utama notebook:
- membangun baseline Linear SVM yang kuat
- membandingkannya dengan Logistic Regression
- menyajikan evaluasi yang siap dipakai untuk laporan akademik dan advisor meeting

## SECTION 1 - Project Overview

### Tujuan eksperimen
Tujuan eksperimen ini adalah membangun baseline klasifikasi sentimen berbasis teks yang kuat, stabil, dan mudah diinterpretasikan untuk dataset review UMKM Indonesia pada proyek Sigap.ai. Hasil eksperimen akan dipakai sebagai benchmark untuk membandingkan model lanjutan berikutnya.

### Mengapa Linear SVM digunakan pada text classification
Linear SVM sangat sering menjadi baseline kuat untuk teks karena:
- efektif pada feature sparse berdimensi tinggi seperti TF-IDF
- memiliki margin maximization yang membantu generalisasi
- sering unggul pada problem klasifikasi teks dibanding model linear lain
- tetap relatif mudah dijelaskan melalui koefisien fitur pada kernel linear

### Perbedaan Logistic Regression dan SVM
Logistic Regression memodelkan probabilitas kelas secara langsung, sedangkan SVM berfokus pada pemisahan margin antar kelas.
Dalam praktik text classification:
- Logistic Regression sering lebih probabilistik dan lebih mudah diubah ke confidence score
- Linear SVM sering memberikan batas keputusan yang lebih tegas
- keduanya cocok sebagai baseline, tetapi SVM sering lebih kuat pada feature TF-IDF

## SECTION 2 - Import Library

Library yang digunakan dibatasi pada stack yang umum dipakai untuk eksperimen machine learning terstruktur:
- Pandas
- Numpy
- Matplotlib
- Seaborn
- Scikit-Learn
- Joblib

In [ ]:
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, learning_curve, validation_curve
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['font.size'] = 11
pd.set_option('display.max_colwidth', 250)
pd.set_option('display.max_columns', 100)

## SECTION 3 - Load Dataset

Notebook ini mencoba menemukan file dataset secara otomatis agar tetap fleksibel saat dijalankan di Google Colab maupun pada workspace lokal.

File yang didukung:
- `df_train_final.csv` atau `train_final.csv`
- `df_validation_final.csv` atau `validation_final.csv`
- `df_test_final.csv` atau `test_final.csv`

In [ ]:
# Optional jika dataset disimpan di Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

DATASET_FILE_CANDIDATES = {
    'train': ['df_train_final.csv', 'train_final.csv'],
    'validation': ['df_validation_final.csv', 'validation_final.csv'],
    'test': ['df_test_final.csv', 'test_final.csv'],
}

SEARCH_DIRS = [
    Path('/content'),
    Path('/content/data'),
    Path('/content/dataset'),
    Path('/content/drive/MyDrive'),
    Path.cwd(),
    Path.cwd() / 'dataset',
    Path.cwd() / 'ai',
    Path.cwd() / 'ai' / 'dataset',
    Path.cwd() / 'ai' / 'dataset' / 'processed-dataset',
]

def find_dataset_file(split_name):
    for base_dir in SEARCH_DIRS:
        for filename in DATASET_FILE_CANDIDATES[split_name]:
            candidate = base_dir / filename
            if candidate.exists():
                return candidate
    return None

train_path = find_dataset_file('train')
val_path = find_dataset_file('validation')
test_path = find_dataset_file('test')

if train_path is None or val_path is None or test_path is None:
    raise FileNotFoundError(
        'Dataset tidak ditemukan. Pastikan file train/validation/test final tersedia di lokasi yang dapat diakses notebook.'
    )

print('Train path     :', train_path)
print('Validation path:', val_path)
print('Test path      :', test_path)

In [ ]:
def load_split(path):
    df = pd.read_csv(path, encoding='utf-8-sig')
    expected_columns = [
        'Review_Text',
        'Review_Text_Processed',
        'Rating_Score',
        'Business_Category',
        'Sentiment_Label',
        'Review_Aspect',
        'Crisis_Flag',
        'Is_Sarcasm',
    ]
    missing = [c for c in expected_columns if c not in df.columns]
    if missing:
        raise ValueError(f'{path.name} missing columns: {missing}')
    return df

train_df = load_split(train_path)
val_df = load_split(val_path)
test_df = load_split(test_path)

for split_name, df in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    print(f'\n{split_name} shape: {df.shape}')
    print('Label distribution:')
    display(df['Sentiment_Label'].value_counts().to_frame('count'))
    print('Sample data:')
    display(df[['Review_Text_Processed', 'Sentiment_Label']].head(5))

## SECTION 4 - Label Encoding

Pada tahap ini label distandarkan menjadi `Positive`, `Neutral`, `Negative`, kemudian diencode agar kompatibel dengan model klasifikasi.
Mapping label disimpan agar interpretasi hasil tetap transparan.

In [ ]:
def standardize_label(label):
    label = str(label).strip()
    if label.lower() == 'netral':
        return 'Neutral'
    return label

for df in [train_df, val_df, test_df]:
    df['Sentiment_Label'] = df['Sentiment_Label'].apply(standardize_label)

label_order = ['Negative', 'Neutral', 'Positive']
label_encoder = LabelEncoder()
label_encoder.fit(label_order)

label_mapping = {label: int(label_encoder.transform([label])[0]) for label in label_encoder.classes_}
inverse_label_mapping = {v: k for k, v in label_mapping.items()}

display(pd.DataFrame({'label': list(label_mapping.keys()), 'encoded_value': list(label_mapping.values())}))

train_y = label_encoder.transform(train_df['Sentiment_Label'])
val_y = label_encoder.transform(val_df['Sentiment_Label'])
test_y = label_encoder.transform(test_df['Sentiment_Label'])

## SECTION 5 - TF-IDF Feature Engineering

Eksperimen TF-IDF dilakukan untuk membandingkan representasi teks yang berbeda sebelum model final dibangun.

Konfigurasi yang diuji:
- Unigram
- Bigram
- `max_features = 5000`
- `max_features = 10000`
- `max_features = 20000`

In [ ]:
X_train = train_df['Review_Text_Processed'].fillna('').astype(str)
X_val = val_df['Review_Text_Processed'].fillna('').astype(str)
X_test = test_df['Review_Text_Processed'].fillna('').astype(str)

def build_pipeline(tfidf_params, model_params=None):
    if model_params is None:
        model_params = {}
    vectorizer = TfidfVectorizer(
        lowercase=True,
        strip_accents='unicode',
        sublinear_tf=True,
        norm='l2',
        **tfidf_params,
    )
    classifier = LinearSVC(random_state=42, **model_params)
    return Pipeline([
        ('tfidf', vectorizer),
        ('svm', classifier),
    ])

def evaluate_pipeline(pipeline, x_train, y_train, x_eval, y_eval):
    pipeline.fit(x_train, y_train)
    pred = pipeline.predict(x_eval)
    return {
        'accuracy': accuracy_score(y_eval, pred),
        'precision_macro': precision_score(y_eval, pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_eval, pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_eval, pred, average='macro', zero_division=0),
        'pred': pred,
        'pipeline': pipeline,
    }

experiment_rows = []
tfidf_configs = [
    ('Unigram_5k', {'ngram_range': (1, 1), 'max_features': 5000, 'min_df': 2, 'max_df': 0.95}),
    ('Unigram_10k', {'ngram_range': (1, 1), 'max_features': 10000, 'min_df': 2, 'max_df': 0.95}),
    ('Unigram_20k', {'ngram_range': (1, 1), 'max_features': 20000, 'min_df': 2, 'max_df': 0.95}),
    ('Bigram_5k', {'ngram_range': (1, 2), 'max_features': 5000, 'min_df': 2, 'max_df': 0.95}),
    ('Bigram_10k', {'ngram_range': (1, 2), 'max_features': 10000, 'min_df': 2, 'max_df': 0.95}),
    ('Bigram_20k', {'ngram_range': (1, 2), 'max_features': 20000, 'min_df': 2, 'max_df': 0.95}),
]

for exp_name, tfidf_params in tfidf_configs:
    pipe = build_pipeline(tfidf_params)
    result = evaluate_pipeline(pipe, X_train, train_y, X_val, val_y)
    experiment_rows.append({
        'experiment': exp_name,
        'ngram_range': str(tfidf_params['ngram_range']),
        'max_features': tfidf_params['max_features'],
        'accuracy': result['accuracy'],
        'precision_macro': result['precision_macro'],
        'recall_macro': result['recall_macro'],
        'f1_macro': result['f1_macro'],
        'vocab_size': len(result['pipeline'].named_steps['tfidf'].vocabulary_),
    })

experiment_results = pd.DataFrame(experiment_rows).sort_values('f1_macro', ascending=False).reset_index(drop=True)
display(experiment_results)

fig, ax = plt.subplots(figsize=(14, 7))
plot_df = experiment_results.sort_values('f1_macro', ascending=True)
sns.barplot(data=plot_df, x='f1_macro', y='experiment', palette='viridis', ax=ax)
ax.set_title('TF-IDF Configuration Comparison on Validation Set')
ax.set_xlabel('Macro F1 Score')
ax.set_ylabel('Configuration')
for i, v in enumerate(plot_df['f1_macro']):
    ax.text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

best_tfidf_row = experiment_results.iloc[0]
BEST_TFIDF_PARAMS = {
    'ngram_range': eval(best_tfidf_row['ngram_range']),
    'max_features': int(best_tfidf_row['max_features']),
    'min_df': 2,
    'max_df': 0.95,
}
print(BEST_TFIDF_PARAMS)

## SECTION 6 - Baseline Linear SVM

Baseline model dibangun menggunakan konfigurasi TF-IDF terbaik dari Section 5 dan `LinearSVC()` default.
Evaluasi dilakukan pada validation set untuk mendapatkan baseline yang kredibel sebelum tuning.

In [ ]:
baseline_svm = build_pipeline(BEST_TFIDF_PARAMS)
baseline_svm.fit(X_train, train_y)

val_pred_baseline = baseline_svm.predict(X_val)

baseline_validation_metrics = pd.DataFrame([{
    'split': 'validation',
    'accuracy': accuracy_score(val_y, val_pred_baseline),
    'precision_macro': precision_score(val_y, val_pred_baseline, average='macro', zero_division=0),
    'recall_macro': recall_score(val_y, val_pred_baseline, average='macro', zero_division=0),
    'f1_macro': f1_score(val_y, val_pred_baseline, average='macro', zero_division=0),
}])
display(baseline_validation_metrics)

print(classification_report(val_y, val_pred_baseline, target_names=label_encoder.classes_, digits=4, zero_division=0))

## SECTION 7 - Hyperparameter Tuning

Hyperparameter tuning dilakukan menggunakan GridSearchCV pada training set.

Grid yang diuji:
- `C = [0.01, 0.1, 1, 10, 100]`
- `loss = ['hinge', 'squared_hinge']`
- `class_weight = [None, 'balanced']`
- `max_iter = [2000, 5000]`

Metric utama yang digunakan adalah `F1 Macro`.

In [ ]:
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

tuning_model = build_pipeline(BEST_TFIDF_PARAMS)
param_grid = {
    'svm__C': [0.01, 0.1, 1, 10, 100],
    'svm__loss': ['hinge', 'squared_hinge'],
    'svm__class_weight': [None, 'balanced'],
    'svm__max_iter': [2000, 5000],
}

grid_search = GridSearchCV(
    estimator=tuning_model,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=cv_strategy,
    n_jobs=-1,
    verbose=1,
    refit=True,
    return_train_score=True,
)

grid_search.fit(X_train, train_y)

print('Best Parameters:')
print(grid_search.best_params_)
print('Best CV F1 Macro:', grid_search.best_score_)

grid_results = pd.DataFrame(grid_search.cv_results_)
cols_to_show = [
    'rank_test_score',
    'mean_test_score',
    'std_test_score',
    'mean_train_score',
    'std_train_score',
    'param_svm__C',
    'param_svm__loss',
    'param_svm__class_weight',
    'param_svm__max_iter',
]
grid_results_table = grid_results[cols_to_show].sort_values('rank_test_score').reset_index(drop=True)
display(grid_results_table)

fig, ax = plt.subplots(figsize=(14, 8))
top_grid = grid_results_table.head(20).copy()
top_grid['config'] = (
    'C=' + top_grid['param_svm__C'].astype(str) +
    ' | ' + top_grid['param_svm__loss'].astype(str) +
    ' | ' + top_grid['param_svm__class_weight'].astype(str) +
    ' | iter=' + top_grid['param_svm__max_iter'].astype(str)
)
sns.barplot(
    data=top_grid.sort_values('mean_test_score', ascending=True),
    x='mean_test_score',
    y='config',
    palette='mako',
    ax=ax,
)
ax.set_title('Top GridSearchCV Results by Macro F1')
ax.set_xlabel('Mean CV Macro F1')
ax.set_ylabel('Configuration')
plt.tight_layout()
plt.show()

## SECTION 8 - Final Model Training

Final model dibangun menggunakan parameter terbaik dari GridSearchCV. Model kemudian dilatih ulang pada training set agar seluruh data latih berkontribusi pada parameter final.

In [ ]:
final_model = build_pipeline(BEST_TFIDF_PARAMS)
final_model.set_params(**grid_search.best_params_)
final_model.fit(X_train, train_y)

final_vectorizer = final_model.named_steps['tfidf']
final_classifier = final_model.named_steps['svm']

print('Final model fitted successfully.')
print('Vocabulary size:', len(final_vectorizer.vocabulary_))

## SECTION 9 - Evaluation

Evaluasi dilakukan pada validation set dan test set dengan metrik berikut:
- Accuracy
- Precision
- Recall
- F1 Score

Selain metrik agregat, notebook juga menampilkan classification report lengkap untuk masing-masing split.

In [ ]:
def evaluate_model(model, x, y, split_name):
    pred = model.predict(x)
    metrics = {
        'split': split_name,
        'accuracy': accuracy_score(y, pred),
        'precision_macro': precision_score(y, pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y, pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y, pred, average='macro', zero_division=0),
    }
    report_dict = classification_report(
        y,
        pred,
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report_dict).T.reset_index().rename(columns={'index': 'label'})
    return metrics, pred, report_df

val_metrics, val_pred, val_report_df = evaluate_model(final_model, X_val, val_y, 'validation')
test_metrics, test_pred, test_report_df = evaluate_model(final_model, X_test, test_y, 'test')

metrics_df = pd.DataFrame([val_metrics, test_metrics])
display(metrics_df)
display(val_report_df)
display(test_report_df)

print('Validation Classification Report')
print(classification_report(val_y, val_pred, target_names=label_encoder.classes_, digits=4, zero_division=0))

print('Test Classification Report')
print(classification_report(test_y, test_pred, target_names=label_encoder.classes_, digits=4, zero_division=0))

## SECTION 10 - Visualization

Visualisasi dibuat untuk membantu interpretasi performa model secara akademik dan presentatif.

Visualisasi yang disediakan:
1. Confusion Matrix Heatmap
2. Learning Curve
3. Validation Curve
4. Classification Report Heatmap
5. Prediction Distribution
6. Actual vs Prediction Distribution

In [ ]:
class_names = list(label_encoder.classes_)

# 1. Confusion Matrix Heatmap
cm = confusion_matrix(test_y, test_pred)
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_title('Confusion Matrix Heatmap - Test Set')
ax.set_xlabel('Predicted Label')
ax.set_ylabel('Actual Label')
plt.tight_layout()
plt.show()

# 2. Learning Curve
train_sizes, train_scores, valid_scores = learning_curve(
    estimator=final_model,
    X=X_train,
    y=train_y,
    train_sizes=np.linspace(0.1, 1.0, 5),
    cv=cv_strategy,
    scoring='f1_macro',
    n_jobs=-1,
    shuffle=True,
    random_state=42,
)
train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
valid_mean = valid_scores.mean(axis=1)
valid_std = valid_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(train_sizes, train_mean, marker='o', label='Training Score')
ax.plot(train_sizes, valid_mean, marker='o', label='Cross-Validation Score')
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15)
ax.fill_between(train_sizes, valid_mean - valid_std, valid_mean + valid_std, alpha=0.15)
ax.set_title('Learning Curve - Linear SVM')
ax.set_xlabel('Training Samples')
ax.set_ylabel('Macro F1 Score')
ax.legend()
plt.tight_layout()
plt.show()

# 3. Validation Curve
c_values = np.array([0.01, 0.1, 1, 10, 100])
val_train_scores, val_test_scores = validation_curve(
    estimator=final_model,
    X=X_train,
    y=train_y,
    param_name='svm__C',
    param_range=c_values,
    cv=cv_strategy,
    scoring='f1_macro',
    n_jobs=-1,
)

fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(c_values, val_train_scores.mean(axis=1), marker='o', label='Training Score')
ax.plot(c_values, val_test_scores.mean(axis=1), marker='o', label='Validation Score')
ax.set_xscale('log')
ax.set_title('Validation Curve - C Parameter')
ax.set_xlabel('C')
ax.set_ylabel('Macro F1 Score')
ax.legend()
plt.tight_layout()
plt.show()

# 4. Classification Report Heatmap
report_heatmap_df = pd.DataFrame({
    'precision': classification_report(test_y, test_pred, target_names=label_encoder.classes_, output_dict=True, zero_division=0).keys()
})
test_report_only_classes = pd.DataFrame(classification_report(
    test_y, test_pred, target_names=label_encoder.classes_, output_dict=True, zero_division=0
)).T.loc[label_encoder.classes_, ['precision', 'recall', 'f1-score']]

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(test_report_only_classes, annot=True, cmap='YlGnBu', vmin=0, vmax=1, ax=ax)
ax.set_title('Classification Report Heatmap - Test Set')
ax.set_xlabel('Metric')
ax.set_ylabel('Class')
plt.tight_layout()
plt.show()

# 5. Prediction Distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
val_pred_dist = pd.Series(val_pred).map(inverse_label_mapping).value_counts().reindex(class_names).fillna(0)
test_pred_dist = pd.Series(test_pred).map(inverse_label_mapping).value_counts().reindex(class_names).fillna(0)
sns.barplot(x=val_pred_dist.index, y=val_pred_dist.values, ax=axes[0], palette='viridis')
axes[0].set_title('Prediction Distribution - Validation')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('Count')
sns.barplot(x=test_pred_dist.index, y=test_pred_dist.values, ax=axes[1], palette='viridis')
axes[1].set_title('Prediction Distribution - Test')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.show()

# 6. Actual vs Prediction Distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
val_actual_dist = pd.Series(val_y).map(inverse_label_mapping).value_counts().reindex(class_names).fillna(0)
test_actual_dist = pd.Series(test_y).map(inverse_label_mapping).value_counts().reindex(class_names).fillna(0)

def grouped_distribution(ax, actual_dist, pred_dist, title):
    x = np.arange(len(class_names))
    width = 0.35
    ax.bar(x - width / 2, actual_dist.values, width=width, label='Actual')
    ax.bar(x + width / 2, pred_dist.values, width=width, label='Predicted')
    ax.set_xticks(x)
    ax.set_xticklabels(class_names)
    ax.set_title(title)
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    ax.legend()
    for i, value in enumerate(actual_dist.values):
        ax.text(i - width / 2, value + 5, str(int(value)), ha='center', va='bottom', fontsize=9)
    for i, value in enumerate(pred_dist.values):
        ax.text(i + width / 2, value + 5, str(int(value)), ha='center', va='bottom', fontsize=9)

grouped_distribution(axes[0], val_actual_dist, val_pred_dist, 'Actual vs Prediction - Validation')
grouped_distribution(axes[1], test_actual_dist, test_pred_dist, 'Actual vs Prediction - Test')
plt.tight_layout()
plt.show()

## SECTION 11 - Error Analysis

Error analysis difokuskan pada contoh yang salah klasifikasi di test set karena test set adalah representasi paling ketat dari generalisasi model.

Tabel berikut menampilkan 50 contoh salah klasifikasi dengan kolom:
- `Review_Text`
- `Actual_Label`
- `Predicted_Label`

In [ ]:
test_predictions_df = pd.DataFrame({
    'Review_Text': test_df['Review_Text'],
    'Review_Text_Processed': test_df['Review_Text_Processed'],
    'Actual_Label': pd.Series(test_y).map(inverse_label_mapping),
    'Predicted_Label': pd.Series(test_pred).map(inverse_label_mapping),
    'Crisis_Flag': test_df['Crisis_Flag'].values,
    'Is_Sarcasm': test_df['Is_Sarcasm'].values,
    'Business_Category': test_df['Business_Category'].values,
    'Review_Aspect': test_df['Review_Aspect'].values,
})

misclassified_df = test_predictions_df[test_predictions_df['Actual_Label'] != test_predictions_df['Predicted_Label']].copy()
error_examples = misclassified_df[['Review_Text', 'Actual_Label', 'Predicted_Label']].head(50)

display(error_examples)

print('Jumlah salah klasifikasi:', len(misclassified_df))
print('Error rate:', round(len(misclassified_df) / len(test_predictions_df) * 100, 2), '%')
print('\nPola kesalahan utama:')
print('- Review pendek atau ringkas lebih sulit dipisahkan antar kelas.')
print('- Ulasan netral sering tertukar dengan positif atau negatif jika mengandung kata evaluatif yang ambigu.')
print('- Kasus sarcasm cenderung meningkatkan risiko salah klasifikasi.')
print('- Review dengan konteks domain sangat spesifik dapat diprediksi kurang stabil jika kosakata latih terbatas.')

confusion_pairs = misclassified_df.groupby(['Actual_Label', 'Predicted_Label']).size().sort_values(ascending=False).to_frame('count')
display(confusion_pairs.head(10))

## SECTION 12 - Explainable AI

Koefisien Linear SVM digunakan untuk mengekstrak kata-kata yang paling berkontribusi pada masing-masing kelas.

Interpretasi yang digunakan:
- kata dengan bobot tertinggi pada kelas `Positive` dianggap paling mendukung prediksi positif
- kata dengan bobot tertinggi pada kelas `Negative` dianggap paling mendukung prediksi negatif
- kata dengan bobot tertinggi pada kelas `Neutral` dianggap paling mendukung prediksi netral

In [ ]:
feature_names = np.array(final_vectorizer.get_feature_names_out())
coef = final_classifier.coef_
class_names = list(label_encoder.classes_)

def resolve_class_index(class_name):
    encoded_value = label_mapping[class_name]
    return np.where(final_classifier.classes_ == encoded_value)[0][0]

explainable_tables = {}
for class_name in ['Positive', 'Neutral', 'Negative']:
    class_idx = resolve_class_index(class_name)
    class_coef = coef[class_idx]
    top_idx = np.argsort(class_coef)[-20:][::-1]
    table = pd.DataFrame({
        'feature': feature_names[top_idx],
        'coefficient': class_coef[top_idx],
    })
    explainable_tables[class_name] = table
    display(Markdown(f'### Top 20 Features for {class_name}'))
    display(table)

fig, axes = plt.subplots(3, 1, figsize=(14, 18))
colors = {'Positive': '#2ca02c', 'Neutral': '#ff7f0e', 'Negative': '#d62728'}
for ax, class_name in zip(axes, ['Positive', 'Neutral', 'Negative']):
    table = explainable_tables[class_name].sort_values('coefficient', ascending=True)
    ax.barh(table['feature'], table['coefficient'], color=colors[class_name])
    ax.set_title(f'Top 20 Features - {class_name}')
    ax.set_xlabel('Weight')
    ax.set_ylabel('Word')
plt.tight_layout()
plt.show()

print('Interpretasi bisnis:')
print('- Fitur positif yang kuat menunjukkan kata yang menandakan kepuasan, kualitas, dan pengalaman baik.')
print('- Fitur negatif yang kuat menunjukkan kata yang mengindikasikan keluhan, gangguan, atau kekecewaan.')
print('- Fitur netral biasanya merepresentasikan kata evaluatif yang tidak ekstrem atau konteks informasional.')

## SECTION 13 - Business Analysis

Analisis bisnis digunakan untuk melihat apakah model bekerja konsisten pada subset data tertentu.

Subgroup yang dianalisis:
- `Crisis_Flag`
- `Is_Sarcasm`
- `Review_Aspect`
- `Business_Category`

In [ ]:
def subgroup_metrics(df_source, actual_array, pred_array, group_col, min_count=1):
    temp = df_source[[group_col]].copy()
    temp['Actual'] = actual_array
    temp['Predicted'] = pred_array
    rows = []
    for group_value, group_df in temp.groupby(group_col):
        if len(group_df) < min_count:
            continue
        rows.append({
            group_col: group_value,
            'count': len(group_df),
            'accuracy': accuracy_score(group_df['Actual'], group_df['Predicted']),
            'precision_macro': precision_score(group_df['Actual'], group_df['Predicted'], average='macro', zero_division=0),
            'recall_macro': recall_score(group_df['Actual'], group_df['Predicted'], average='macro', zero_division=0),
            'f1_macro': f1_score(group_df['Actual'], group_df['Predicted'], average='macro', zero_division=0),
        })
    return pd.DataFrame(rows).sort_values('f1_macro', ascending=False)

crisis_metrics = subgroup_metrics(test_df, test_y, test_pred, 'Crisis_Flag')
sarcasm_metrics = subgroup_metrics(test_df, test_y, test_pred, 'Is_Sarcasm')
aspect_metrics = subgroup_metrics(test_df, test_y, test_pred, 'Review_Aspect', min_count=10)
category_metrics = subgroup_metrics(test_df, test_y, test_pred, 'Business_Category', min_count=10)

display(crisis_metrics)
display(sarcasm_metrics)
display(aspect_metrics)
display(category_metrics)

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
sns.barplot(data=crisis_metrics, x='Crisis_Flag', y='f1_macro', ax=axes[0, 0], palette='coolwarm')
axes[0, 0].set_title('F1 Macro by Crisis_Flag')
axes[0, 0].set_xlabel('Crisis_Flag')
axes[0, 0].set_ylabel('F1 Macro')

sns.barplot(data=sarcasm_metrics, x='Is_Sarcasm', y='f1_macro', ax=axes[0, 1], palette='magma')
axes[0, 1].set_title('F1 Macro by Is_Sarcasm')
axes[0, 1].set_xlabel('Is_Sarcasm')
axes[0, 1].set_ylabel('F1 Macro')

aspect_plot = aspect_metrics.sort_values('count', ascending=False).head(10)
sns.barplot(data=aspect_plot, y='Review_Aspect', x='f1_macro', ax=axes[1, 0], palette='viridis')
axes[1, 0].set_title('Top Review_Aspect by Count - F1 Macro')
axes[1, 0].set_xlabel('F1 Macro')
axes[1, 0].set_ylabel('Review_Aspect')

category_plot = category_metrics.sort_values('count', ascending=False).head(10)
sns.barplot(data=category_plot, y='Business_Category', x='f1_macro', ax=axes[1, 1], palette='viridis')
axes[1, 1].set_title('Top Business_Category by Count - F1 Macro')
axes[1, 1].set_xlabel('F1 Macro')
axes[1, 1].set_ylabel('Business_Category')

plt.tight_layout()
plt.show()

business_summary = pd.concat([
    crisis_metrics.assign(group_type='Crisis_Flag'),
    sarcasm_metrics.assign(group_type='Is_Sarcasm'),
    aspect_metrics.assign(group_type='Review_Aspect'),
    category_metrics.assign(group_type='Business_Category'),
], ignore_index=True)
display(business_summary)

## SECTION 14 - Model Comparison

Bagian ini membandingkan hasil Linear SVM dengan Logistic Regression.

Notebook akan mencoba mengimpor hasil Logistic Regression dari file hasil eksperimen sebelumnya.
Jika file belum tersedia, notebook akan menjalankan fallback Logistic Regression baseline agar perbandingan tetap dapat dibuat.

In [ ]:
def summarize_metrics(split_name, y_true, y_pred):
    return {
        'split': split_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
    }

lr_metrics_path_candidates = [
    Path('results') / 'metrics_logistic_regression.csv',
    Path('metrics_logistic_regression.csv'),
    Path('ai') / 'results' / 'metrics_logistic_regression.csv',
]

lr_metrics_df = None
for candidate in lr_metrics_path_candidates:
    if candidate.exists():
        lr_metrics_df = pd.read_csv(candidate)
        break

if lr_metrics_df is None:
    print('Logistic Regression metrics file not found. Running fallback Logistic Regression baseline for comparison.')
    lr_model = Pipeline([
        ('tfidf', TfidfVectorizer(
            lowercase=True,
            strip_accents='unicode',
            sublinear_tf=True,
            norm='l2',
            **BEST_TFIDF_PARAMS,
        )),
        ('logreg', LogisticRegression(random_state=42, max_iter=1000)),
    ])
    lr_model.fit(X_train, train_y)
    lr_val_pred = lr_model.predict(X_val)
    lr_test_pred = lr_model.predict(X_test)
    lr_metrics_df = pd.DataFrame([
        summarize_metrics('validation', val_y, lr_val_pred),
        summarize_metrics('test', test_y, lr_test_pred),
    ])

svm_comparison_metrics = pd.DataFrame([
    summarize_metrics('validation', val_y, val_pred),
    summarize_metrics('test', test_y, test_pred),
])

display(lr_metrics_df)
display(svm_comparison_metrics)

compare_table = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1'],
    'Logistic Regression': [
        float(lr_metrics_df.loc[lr_metrics_df['split'] == 'test', 'accuracy'].iloc[0]),
        float(lr_metrics_df.loc[lr_metrics_df['split'] == 'test', 'precision_macro'].iloc[0]),
        float(lr_metrics_df.loc[lr_metrics_df['split'] == 'test', 'recall_macro'].iloc[0]),
        float(lr_metrics_df.loc[lr_metrics_df['split'] == 'test', 'f1_macro'].iloc[0]),
    ],
    'Linear SVM': [
        float(svm_comparison_metrics.loc[svm_comparison_metrics['split'] == 'test', 'accuracy'].iloc[0]),
        float(svm_comparison_metrics.loc[svm_comparison_metrics['split'] == 'test', 'precision_macro'].iloc[0]),
        float(svm_comparison_metrics.loc[svm_comparison_metrics['split'] == 'test', 'recall_macro'].iloc[0]),
        float(svm_comparison_metrics.loc[svm_comparison_metrics['split'] == 'test', 'f1_macro'].iloc[0]),
    ],
})
display(compare_table)

fig, ax = plt.subplots(figsize=(10, 6))
compare_plot = compare_table.melt(id_vars='Metric', var_name='Model', value_name='Score')
sns.barplot(data=compare_plot, x='Metric', y='Score', hue='Model', ax=ax, palette='Set2')
ax.set_title('Logistic Regression vs Linear SVM')
ax.set_xlabel('Metric')
ax.set_ylabel('Score')
plt.tight_layout()
plt.show()

print('Analisis:')
print('- Linear SVM cenderung unggul ketika margin pemisah antar kelas pada ruang TF-IDF lebih jelas.')
print('- Logistic Regression bisa lebih stabil jika probabilitas yang halus lebih penting, tetapi tidak selalu lebih kuat pada teks sparse.')
print('- Jika SVM lebih baik di F1 Macro, itu biasanya menandakan pemisahan kelas yang lebih tegas pada feature sparse.')

## SECTION 15 - Model Saving

Agar model dapat digunakan kembali tanpa retraining, notebook menyimpan tiga artefak utama:
- `svm_model.joblib`
- `tfidf_vectorizer.joblib`
- `label_encoder.joblib`

In [ ]:
artifacts_dir = Path('artifacts_svm')
artifacts_dir.mkdir(exist_ok=True)

joblib.dump(final_classifier, artifacts_dir / 'svm_model.joblib')
joblib.dump(final_vectorizer, artifacts_dir / 'tfidf_vectorizer.joblib')
joblib.dump(label_encoder, artifacts_dir / 'label_encoder.joblib')

print('Saved artifacts to:', artifacts_dir.resolve())

## SECTION 16 - Export Result

Section ini mengekspor hasil eksperimen agar dapat dipakai langsung untuk pelaporan capstone, audit model, dan dokumentasi advisor meeting.

File yang disimpan:
- `metrics_svm.csv`
- `predictions_svm.csv`
- `classification_report_svm.csv`

In [ ]:
results_dir = Path('results_svm')
results_dir.mkdir(exist_ok=True)

metrics_export = pd.DataFrame([val_metrics, test_metrics])
metrics_export.to_csv(results_dir / 'metrics_svm.csv', index=False)

def decision_scores(model, x):
    scores = model.decision_function(x)
    if scores.ndim == 1:
        scores = scores[:, None]
    return scores

val_scores = decision_scores(final_model, X_val)
test_scores = decision_scores(final_model, X_test)

prediction_export = pd.concat([
    pd.DataFrame({
        'split': 'validation',
        'Review_Text_Processed': X_val.values,
        'Actual_Label': pd.Series(val_y).map(inverse_label_mapping).values,
        'Predicted_Label': pd.Series(val_pred).map(inverse_label_mapping).values,
        **{f'score_{cls}': val_scores[:, idx] for idx, cls in enumerate(class_names)},
    }),
    pd.DataFrame({
        'split': 'test',
        'Review_Text_Processed': X_test.values,
        'Actual_Label': pd.Series(test_y).map(inverse_label_mapping).values,
        'Predicted_Label': pd.Series(test_pred).map(inverse_label_mapping).values,
        **{f'score_{cls}': test_scores[:, idx] for idx, cls in enumerate(class_names)},
    }),
], ignore_index=True)
prediction_export.to_csv(results_dir / 'predictions_svm.csv', index=False)

def report_to_export_df(y_true, y_pred, split_name):
    report_dict = classification_report(
        y_true,
        y_pred,
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report_dict).T.reset_index().rename(columns={'index': 'label'})
    report_df['split'] = split_name
    return report_df[['split', 'label', 'precision', 'recall', 'f1-score', 'support']]

classification_report_export = pd.concat([
    report_to_export_df(val_y, val_pred, 'validation'),
    report_to_export_df(test_y, test_pred, 'test'),
], ignore_index=True)
classification_report_export.to_csv(results_dir / 'classification_report_svm.csv', index=False)

display(metrics_export)
display(prediction_export.head())
display(classification_report_export.head())
print('Exported files to:', results_dir.resolve())

## SECTION 17 - Final Conclusion

Ringkasan otomatis berikut dihasilkan dari metrik validation dan test. Bagian ini dapat langsung dipakai sebagai bahan narasi laporan capstone atau diskusi advisor meeting.

In [ ]:
final_conclusion_md = f'''
### Final Conclusion

**Final Test Performance**
- Accuracy: {test_metrics['accuracy']:.4f}
- Precision: {test_metrics['precision_macro']:.4f}
- Recall: {test_metrics['recall_macro']:.4f}
- F1 Score: {test_metrics['f1_macro']:.4f}

**Strength**
- Linear SVM sangat cocok untuk feature sparse seperti TF-IDF.
- Model ini memiliki decision boundary yang tegas dan sering unggul pada klasifikasi teks.
- Koefisien model dapat digunakan untuk interpretasi feature importance.

**Weakness**
- Model linear tetap terbatas dalam menangkap konteks yang sangat kompleks, sarcasm, dan negasi panjang.
- Performa tetap sensitif terhadap kualitas feature engineering dan preprocessing.
- Interpretasi koefisien masih bersifat linear dan tidak menangkap interaksi konteks yang lebih kaya.

**Apakah layak menjadi model final**
- Jika hasil test menunjukkan F1 Macro yang lebih baik dibanding Logistic Regression, model ini layak dijadikan kandidat model final baseline.
- Jika gap performanya kecil, Linear SVM tetap layak dipilih karena interpretabilitas dan stabilitasnya pada text classification.
- Jika kebutuhan bisnis menuntut konteks yang lebih dalam, model ini sebaiknya menjadi baseline kuat sebelum mencoba transformer.
'''

display(Markdown(final_conclusion_md))